# F1 Podium Predictor — Training

Binary XGBoost classifier for the probability a driver finishes on the podium (P≤3).
Thin orchestration: all logic lives in the tested `f1pred` package (see
`specs/003-ml-model`). Run top-to-bottom; it downloads FastF1 seasons (cached in
`.fastf1-cache/`), trains, evaluates against the grid-top-3 baseline, shows SHAP,
and uploads a versioned artifact + model card to S3.

Prereq: `pip install -e ".[dev,notebook]"` from `ml/`, plus AWS creds for the
upload step. FastF1's cold download is rate-limited (500 calls/h), so the first
run over a multi-season window may need to resume across hourly windows; cached
sessions are free on re-run.

In [1]:
# Run from the ml/ project root so the relative .fastf1-cache/ and artifacts/
# paths resolve the same way whether launched via nbconvert or from notebooks/.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())

cwd: /Users/martinschweiger/Developer/personal/F1-project/ml


In [2]:
import logging

import fastf1

from f1pred.data import fastf1_load_race, fastf1_rounds_for_year, load_seasons
from f1pred.pipeline import run_pipeline

logging.basicConfig(level=logging.INFO)
MODEL_VERSION = "0.1.0"
# Temporal split: train ≤ TRAIN_MAX_YEAR, validate on VAL_YEAR, test on TEST_YEAR.
# FIRST_YEAR bounds the FastF1 download window (cold pull is rate-limited to
# 500 calls/h, so we use a focused 2022–2025 window rather than 2018+).
FIRST_YEAR = 2022
TRAIN_MAX_YEAR, VAL_YEAR, TEST_YEAR = 2023, 2024, 2025

/Users/martinschweiger/Developer/personal/F1-project/ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load historical races (FastF1)

First run downloads + caches; later runs are offline. Missing sessions are
skipped and logged (no silent imputation).

In [3]:
years = range(FIRST_YEAR, TEST_YEAR + 1)
races = load_seasons(
    years,
    rounds_for_year=fastf1_rounds_for_year,
    load_race=fastf1_load_race,
)
print(f"{len(races)} race-driver rows across {races['year'].nunique()} seasons")
races.head()

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]


INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Race [v3.8.3]


req            INFO 	Using cached data for session_info


INFO:fastf1.fastf1.req:Using cached data for session_info


req            INFO 	Using cached data for driver_info


...

INFO:fastf1.fastf1.core:Processing timing data...


core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']


INFO:fastf1.fastf1.core:Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']


1838 race-driver rows across 4 seasons


,year,round,driver,constructor,circuit,grid_position,quali_gap_to_pole_s,is_wet,points,finish_position
0,2022,1,LEC,Ferrari,Bahrain Grand Prix,1.0,0.000,False,26.0,1.0
1,2022,1,SAI,Ferrari,Bahrain Grand Prix,3.0,0.129,False,18.0,2.0
2,2022,1,HAM,Mercedes,Bahrain Grand Prix,5.0,0.490,False,15.0,3.0
3,2022,1,RUS,Mercedes,Bahrain Grand Prix,9.0,0.694,False,12.0,4.0
4,2022,1,MAG,Haas F1 Team,Bahrain Grand Prix,7.0,0.903,False,10.0,5.0


## 2. Run the pipeline (features → split → train → eval)

Temporal split, leakage-safe pre-race features, XGBoost with class weights.

In [4]:
result = run_pipeline(
    races,
    train_max_year=TRAIN_MAX_YEAR,
    val_year=VAL_YEAR,
    test_year=TEST_YEAR,
    version=MODEL_VERSION,
    fastf1_version=fastf1.__version__,
)
print(f"train rows: {result.n_train}  test rows: {result.n_test}")

train rows: 845  test rows: 467


## 3. Metrics vs. baseline ("podium = grid ≤ 3")

In [5]:
import pandas as pd

pd.DataFrame({"model": result.metrics, "baseline": result.baseline}).T[
    ["accuracy", "log_loss", "roc_auc"]
]

,accuracy,log_loss,roc_auc
model,0.865096,0.283088,0.934142
baseline,0.922912,1.065008,0.852215


In [6]:
from f1pred.evaluate import calibration_figure, confusion_figure
from f1pred.features import build_features
from f1pred.schema import FEATURE_NAMES
from f1pred.split import temporal_split
from f1pred.target import podium_label

# Rebuild the test fold to render the figures.
_feat = build_features(races)
_data = _feat.assign(
    podium=podium_label(races, position_col="finish_position").reindex(_feat.index)
)
_test = temporal_split(
    _data, train_max_year=TRAIN_MAX_YEAR, val_year=VAL_YEAR, test_year=TEST_YEAR
).test

confusion_figure(result.metrics)

<Figure size 320x300 with 1 Axes>

In [7]:
calibration_figure(result.model, _test[list(FEATURE_NAMES)], _test["podium"])

<Figure size 400x400 with 1 Axes>

## 4. SHAP — global + a single prediction

In [8]:
from f1pred.explain import importance_figure, one_prediction_figure, one_prediction_shap

importance_figure(result.importance)

<Figure size 500x300 with 1 Axes>

In [9]:
one_prediction_figure(one_prediction_shap(result.model, _test[list(FEATURE_NAMES)]))

<Figure size 500x300 with 1 Axes>

## 5. Publish the artifact + model card to S3

Needs AWS creds. Writes `models/<version>/model.json` + `model_card.md`.

In [10]:
import boto3

from f1pred.artifact import publish
from f1pred.layout import bucket_name

region = "eu-central-1"
account = boto3.client("sts").get_caller_identity()["Account"]
s3 = boto3.client("s3", region_name=region)

out = publish(
    result.model,
    result.card_text,
    version=MODEL_VERSION,
    s3_client=s3,
    bucket=bucket_name(account, region),
)
print("published", out)
print(result.card_text)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


INFO:f1pred.artifact:uploaded model 0.1.0 to s3://f1-data-128663321407-eu-central-1/models/0.1.0/


published artifacts/0.1.0
# Model Card — Podium Predictor `0.1.0`

Binary XGBoost classifier: probability a driver finishes on the podium (P≤3).

## Data

- Source: FastF1 3.8.3.
- Seasons / split (temporal, no shuffle): train ≤2023, val 2024, test 2025.
- Rows: 845 train, 467 test.

## Features (pre-race only — no leakage)

- `grid_position` — mean|SHAP| 1.304
- `driver_form` — mean|SHAP| 1.155
- `quali_gap_to_pole_s` — mean|SHAP| 0.790
- `constructor_form` — mean|SHAP| 0.460
- `track_history` — mean|SHAP| 0.101
- `is_wet` — mean|SHAP| 0.026

## Metrics (test) vs. baseline "podium = grid ≤ 3"

| model | accuracy | log_loss | roc_auc |
| ----- | -------- | -------- | ------- |
| XGBoost | 0.865 | 0.283 | 0.934 |
| baseline | 0.923 | 1.065 | 0.852 |

## Limitations

Imbalanced target (~15% podium); FastF1 gaps dropped, not imputed; no live-session signal yet (Phase 5).

## Cost

Training is local + offline (FastF1 cache); the artifact is a few MB in S3 — ≈0 €
(Constitution IV).

